#Random Probabilistic Model

**Author:** Thor Lemke

This file uses the training split to compute the probability for each individual label, and then uses random number generation to assign labels to the test set based on the probabilities from the training set.  F1 scores are reported, and are low, as expected for a random model.  However, this provides a basis to demonstrate that future models are improvements on random assignment.

In [1]:
from datasets import load_dataset
import numpy as np
from sklearn.metrics import f1_score

In [2]:
# Load the dataset and all splits
ds_simplified = load_dataset("google-research-datasets/go_emotions", "simplified")
ds_train = ds_simplified["train"]
ds_validate = ds_simplified["validation"]
ds_test = ds_simplified["test"]

README.md:   0%|          | 0.00/9.40k [00:00<?, ?B/s]

simplified/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 2.77MB            

simplified/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

simplified/validation-00000-of-00001.par(…): reconstructing file:   0%|          |  0.00B /  350kB            

simplified/validation-00000-of-00001.par(…): downloading bytes:           |  0.00B            

simplified/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  347kB            

simplified/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/43410 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/5426 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/5427 [00:00<?, ? examples/s]

In [11]:
# Find the number of each label present in the training set
training_label_tracker = np.zeros((28,))
for label in ds_train['labels']:
  for val in label:
    training_label_tracker[val] += 1

# Convert these numbers to probabilities
train_points = len(ds_train.unique("id"))
train_prob = training_label_tracker / train_points
print(train_prob)

[0.09513937 0.0536282  0.03609767 0.05689933 0.06770329 0.02504031
 0.03151348 0.05047224 0.01476618 0.0292329  0.04657913 0.01826768
 0.00697996 0.01964985 0.01372956 0.06132228 0.00177378 0.03344851
 0.04805344 0.00377793 0.03642018 0.00255701 0.02557015 0.00352453
 0.01255471 0.03054596 0.02441834 0.32755126]


In [15]:
# Assign labels randomly to the test set
random_predictions = np.zeros((ds_test.num_rows, ds_test.features["labels"].feature.num_classes))

for i in range(ds_test.num_rows):

  samples = np.random.rand(ds_test.features["labels"].feature.num_classes, 1)

  label_added = False
  for j in range(ds_test.features["labels"].feature.num_classes):
    if samples[j] < train_prob[j]:
      label_added = True
      random_predictions[i, j] = 1

  # If no labels were randomly added, assign neutral
  # This ensures that a label is assigned, and is similar to labeling process
  # Note that this will skew toward a generally higher neutral rate on the
  # predictions than the actual data, but is worthwhile for the reasons noted
  # above
  if not label_added:
    random_predictions[i, -1] = 1


In [5]:
# Create a matrix for the actual labels of the test set
def generate_binary_results(data):
  binary_results = np.zeros((data.num_rows, data.features["labels"].feature.num_classes))
  for i in range(data.num_rows):
    labels = data[i]["labels"]
    for label in labels:
      binary_results[i, label] = 1

  return binary_results

test_results = generate_binary_results(ds_test)

In [16]:
# Report individual label and macro F1 scores
running_total_f1 = 0

for i in range(ds_test.features["labels"].feature.num_classes):

  f1 = f1_score(test_results[:, i], random_predictions[:, i])
  running_total_f1 += f1
  print(f"F1 Scores on Test Data for {ds_test.features["labels"].feature.int2str(i)}: {f1}")

print(f"\nMacro F1 Score: {running_total_f1/ds_test.features["labels"].feature.num_classes}")

F1 Scores on Test Data for admiration: 0.09306742640075974
F1 Scores on Test Data for amusement: 0.05328596802841918
F1 Scores on Test Data for anger: 0.035443037974683546
F1 Scores on Test Data for annoyance: 0.03915171288743882
F1 Scores on Test Data for approval: 0.06404657933042213
F1 Scores on Test Data for caring: 0.006993006993006993
F1 Scores on Test Data for confusion: 0.04833836858006042
F1 Scores on Test Data for curiosity: 0.04119850187265917
F1 Scores on Test Data for desire: 0.010638297872340425
F1 Scores on Test Data for disappointment: 0.05901639344262295
F1 Scores on Test Data for disapproval: 0.04040404040404041
F1 Scores on Test Data for disgust: 0.008733624454148471
F1 Scores on Test Data for embarrassment: 0.0
F1 Scores on Test Data for excitement: 0.017937219730941704
F1 Scores on Test Data for fear: 0.011627906976744186
F1 Scores on Test Data for gratitude: 0.06794682422451995
F1 Scores on Test Data for grief: 0.0
F1 Scores on Test Data for joy: 0.021621621621621